In [1]:
#run this to pull content from google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Loading the libraries
import pandas as pd
import numpy as np

# Load in recent data file

In [ ]:
#Loading the Excel File. (Save an excel sheet of the GOGET Main file to GOGET iterations prior to doing this)
goget_folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Snapshots of GOGET main data set/'
goget_xl = pd.ExcelFile(goget_folder + 'GOGET oil & gas extraction sites (2025 02 18 1043).xlsx')

xl = goget_xl

In [ ]:
#setting dataframes with main, P&R
main_data_df = xl.parse('Main data')
production_reserves_df = xl.parse('Reserves and Production')


In [ ]:
#Use this block for any necessary filters: See example of filter for FID year-
  #main_data_df["FID Year"] = main_data_df["FID Year"].astype(str).str.replace(" \(expected\)", "", regex=True)
  #filtered_main_data_df = main_data_df[main_data_df["FID Year"].isin(["2022", "2023"])]
filtered_main_data_df = main_data_df[main_data_df["keep"] == "yes"]

In [ ]:
filtered_main_data_df.head()


,Unit ID,Unit Name,Unit name local script,Country/Area,"Subnational unit (province, state)",Status,Status year,Status source,Latitude,Longitude,...,Wiki name,Wiki URL,Wiki URL local,Fuel type,Unit type,Production data?,Reserves data?,Last updated,Notes,keep
0,OG0000001,Matzen,NaN,Austria,NaN,operating,2023,https://www.offshore-technology.com/marketdata...,48.41667,16.71667,...,Matzen Oil and Gas Field (Austria),https://www.gem.wiki/Matzen_Oil_and_Gas_Field_...,NaN,oil and gas,field,/ 2014 /,2023,2024-09-11,Added to download file 2024 10 16,yes
1,OG0000002,Abalone,Abalone,Brazil,Espírito Santo,operating,2024,ANP Shapefile de Dados 2024,-21.36000,-39.61200,...,"Abalone Oil and Gas Field (Espírito Santo, Bra...",https://www.gem.wiki/Abalone_Oil_and_Gas_Field...,https://www.gem.wiki/Campo_de_Petróleo_e_Gás_A...,oil and gas,field,2023 / 2023 /,2015,2024-11-21,Fields in this block are going to be decomissi...,yes
5,OG0000006,Aguilhada,Aguilhada,Brazil,Sergipe,operating,2023,BMPPGN2023,-10.68600,-36.88900,...,"Aguilhada Oil and Gas Field (Sergipe, Brazil)",https://www.gem.wiki/Aguilhada_Oil_and_Gas_Fie...,https://www.gem.wiki/Campo_de_Petróleo_e_Gás_A...,oil and gas,field,2023 / 2023 /,2015,2021-02-01,NaN,yes
6,OG0000007,Agulha,Agulha,Brazil,Rio Grande do Norte,operating,2022,ANP Stat 2023 Frame 1.3,-4.89900,-36.26200,...,"Agulha Oil and Gas Field (Rio Grande do Norte,...",https://www.gem.wiki/Agulha_Oil_and_Gas_Field_...,https://www.gem.wiki/Campo_de_Petróleo_e_Gás_A...,oil and gas,field,/ 2023 /,2015,2021-02-01,NaN,yes
7,OG0000008,Albacora,Albacora,Brazil,Rio de Janeiro,operating,2024,ANP Shapefile de Dados 2024,-22.13100,-39.96100,...,"Albacora Oil and Gas Field (Rio de Janeiro, Br...",https://www.gem.wiki/Albacora_Oil_and_Gas_Fiel...,https://www.gem.wiki/Campo_de_Petróleo_e_Gás_A...,oil and gas,field,2023 / 2023 /,2017,2024-11-21,NaN,yes


# Add Reserve and production Data to Main Data Frame

In [ ]:
# Convert 'Data year' to integers in the 'production_reserves_df'
production_reserves_df['Data year'] = pd.to_numeric(production_reserves_df['Data year'], errors='coerce').fillna(-1).astype(int)


In [ ]:
#Define functions for getting the most recent value, calculate the total reserves, calculate the total production and define the conversion factor
# Function to get the most recent value based on the criteria
def get_most_recent_value(unit_id, prod_res, units, df):
    # Filter based on Unit ID, Production/reserves, and Units (converted)
    filtered = df[
        (df["Unit ID"] == unit_id) &
        (df["Production/reserves"] == prod_res) &
        (df["Units (converted)"] == units)
    ]

    # Sort by Data year and get the most recent entry
    filtered = filtered.sort_values(by="Data year", ascending=False)
    return filtered.iloc[0]["Quantity (converted)"] if not filtered.empty else np.nan

# Function to calculate the total reserves
def calculate_total_reserves(row):
    if pd.notna(row['Reserves- Hydrocarbons (unspecified)']):
        return row['Reserves- Hydrocarbons (unspecified)']
    else:
        # Convert gas reserves to boe
        gas_in_boe = row['Reserves- Gas'] * conversion_factor if pd.notna(row['Reserves- Gas']) else 0
        oil_reserves = row['Reserves - Oil'] if pd.notna(row['Reserves - Oil']) else 0
        return gas_in_boe + oil_reserves

# Function to calculate the total production
def calculate_total_production(row):
    if pd.notna(row['Production - Hydrocarbons (unspecified)']):
        return row['Production - Hydrocarbons (unspecified)']
    else:
        # Convert gas production to boe
        gas_in_boe = row['Production - Gas'] * conversion_factor if pd.notna(row['Production - Gas']) else 0
        oil_production = row['Production - Oil'] if pd.notna(row['Production - Oil']) else 0
        return gas_in_boe + oil_production

# Conversion factor from million m³ to million boe for gas
conversion_factor = 5.883 / 1000


In [ ]:
# Populate reserves and production columns with the most recent data
filtered_main_data_df["Reserves - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million bbl", production_reserves_df))
filtered_main_data_df["Reserves- Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million m³", production_reserves_df))
filtered_main_data_df["Reserves- Hydrocarbons (unspecified)"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million boe", production_reserves_df))

filtered_main_data_df["Production - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million bbl/y", production_reserves_df))
filtered_main_data_df["Production - Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million m³/y", production_reserves_df))
filtered_main_data_df["Production - Hydrocarbons (unspecified)"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million boe/y", production_reserves_df))

# Calculate total reserves and production
filtered_main_data_df['Reserves- Total (Oil, Gas and Hydrocarbons)'] = filtered_main_data_df.apply(calculate_total_reserves, axis=1)
filtered_main_data_df['Production - Total (Oil, Gas and Hydrocarbons)'] = filtered_main_data_df.apply(calculate_total_production, axis=1)


<ipython-input-10-e0a58fe69be6>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df["Reserves - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million bbl", production_reserves_df))
<ipython-input-10-e0a58fe69be6>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df["Reserves- Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million m³", production_reserves_df))
<ipython-input-1

In [ ]:
filtered_main_data_df.head()

,Unit ID,Unit Name,Unit name local script,Country/Area,"Subnational unit (province, state)",Status,Status year,Status source,Latitude,Longitude,...,Notes,keep,Reserves - Oil,Reserves- Gas,Reserves- Hydrocarbons (unspecified),Production - Oil,Production - Gas,Production - Hydrocarbons (unspecified),"Reserves- Total (Oil, Gas and Hydrocarbons)","Production - Total (Oil, Gas and Hydrocarbons)"
0,OG0000001,Matzen,NaN,Austria,NaN,operating,2023,https://www.offshore-technology.com/marketdata...,48.41667,16.71667,...,Added to download file 2024 10 16,yes,1501.000000,526.688376,NaN,NaN,NaN,NaN,1504.098508,0.000000
1,OG0000002,Abalone,Abalone,Brazil,Espírito Santo,operating,2024,ANP Shapefile de Dados 2024,-21.36000,-39.61200,...,Fields in this block are going to be decomissi...,yes,33.560000,3137.000000,NaN,0.175436,16.151295,NaN,52.014971,0.270454
5,OG0000006,Aguilhada,Aguilhada,Brazil,Sergipe,operating,2023,BMPPGN2023,-10.68600,-36.88900,...,NaN,yes,30.417473,62.886000,NaN,0.002585,0.011133,NaN,30.787431,0.002650
6,OG0000007,Agulha,Agulha,Brazil,Rio Grande do Norte,operating,2022,ANP Stat 2023 Frame 1.3,-4.89900,-36.26200,...,NaN,yes,35.159982,474.460000,NaN,NaN,NaN,NaN,37.951230,0.000000
7,OG0000008,Albacora,Albacora,Brazil,Rio de Janeiro,operating,2024,ANP Shapefile de Dados 2024,-22.13100,-39.96100,...,NaN,yes,4835.090000,111335.020000,NaN,3.482815,39.073287,NaN,5490.073923,3.712683


# Add Keep Tag

In [ ]:
# prompt: filter the filtered_main_data_df for only values where "Reserves- Total (Oil, Gas and Hydrocarbons)" are greater than 25 and/or Production - Total (Oil, Gas and Hydrocarbons) is greater than 1. Add another column thats labelled "Keep?" and enter "yes" for all rows where that criteria is met.

# Filter the DataFrame
filtered_main_data_df['Keep?'] = np.where(
    (filtered_main_data_df['Reserves- Total (Oil, Gas and Hydrocarbons)'] > 25) |
    (filtered_main_data_df['Production - Total (Oil, Gas and Hydrocarbons)'] > 1),
    'yes',
    ''
)

# Display the filtered DataFrame
filtered_main_data_df.head()


<ipython-input-12-186ee2f05814>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df['Keep?'] = np.where(


,Unit ID,Unit Name,Unit name local script,Country/Area,"Subnational unit (province, state)",Status,Status year,Status source,Latitude,Longitude,...,keep,Reserves - Oil,Reserves- Gas,Reserves- Hydrocarbons (unspecified),Production - Oil,Production - Gas,Production - Hydrocarbons (unspecified),"Reserves- Total (Oil, Gas and Hydrocarbons)","Production - Total (Oil, Gas and Hydrocarbons)",Keep?
0,OG0000001,Matzen,NaN,Austria,NaN,operating,2023,https://www.offshore-technology.com/marketdata...,48.41667,16.71667,...,yes,1501.000000,526.688376,NaN,NaN,NaN,NaN,1504.098508,0.000000,yes
1,OG0000002,Abalone,Abalone,Brazil,Espírito Santo,operating,2024,ANP Shapefile de Dados 2024,-21.36000,-39.61200,...,yes,33.560000,3137.000000,NaN,0.175436,16.151295,NaN,52.014971,0.270454,yes
5,OG0000006,Aguilhada,Aguilhada,Brazil,Sergipe,operating,2023,BMPPGN2023,-10.68600,-36.88900,...,yes,30.417473,62.886000,NaN,0.002585,0.011133,NaN,30.787431,0.002650,yes
6,OG0000007,Agulha,Agulha,Brazil,Rio Grande do Norte,operating,2022,ANP Stat 2023 Frame 1.3,-4.89900,-36.26200,...,yes,35.159982,474.460000,NaN,NaN,NaN,NaN,37.951230,0.000000,yes
7,OG0000008,Albacora,Albacora,Brazil,Rio de Janeiro,operating,2024,ANP Shapefile de Dados 2024,-22.13100,-39.96100,...,yes,4835.090000,111335.020000,NaN,3.482815,39.073287,NaN,5490.073923,3.712683,yes


In [ ]:
# prompt: save filtered_main_data_df to /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Snapshots of GOGET main data set/ + GOGET with PR as of + time stamp

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
file_name = f"GOGET with PR as of {timestamp}.xlsx"
output_path = f"/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Snapshots of GOGET main data set/{file_name}"
filtered_main_data_df.to_excel(output_path, index=False)

In [ ]:
# print output path
print(output_path)

/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Snapshots of GOGET main data set/GOGET with PR as of 20250218-182036.xlsx


# Bypass generation

In [ ]:
# Optional: use pregenerated version of the data, to avoid long run time of regenerating
Rendered_file = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Snapshots of GOGET main data set/GOGET with PR as of 20250218-182036.xlsx'
gogetwithpr = pd.read_excel(Rendered_file)

# remove data source, project level reserves, project led columns from df for re-generation
#filtered_main_data_df = filtered_main_data_df.drop(columns=['Data Source', 'Project Level Reserves', 'Project Level Production','Project Led, key data here'])

In [ ]:
# Count blank entries in 'Project' and 'Complex' columns
blank_project_count = gogetwithpr['Project or complex'].isnull().sum()

print(f"Number of blank entries in 'Project' column: {blank_project_count}")


Number of blank entries in 'Project' column: 5064


# Group projects and complexes

In [ ]:
# Sort units by Country/Area, then Project or complex
# Convert 'Country/Area' and 'Project or complex' columns to string type before sorting, and replace NaN with custom string
filledgogetwithpr = gogetwithpr.copy()
filledgogetwithpr['Country/Area'] = filledgogetwithpr['Country/Area'].astype(str)
filledgogetwithpr['Project or complex'] = filledgogetwithpr['Project or complex'].astype(str).replace('nan', 'Unit Only')
gogetwithprsorted = filledgogetwithpr.sort_values(by=['Country/Area', 'Project or complex'])

In [ ]:
# show for qc
gogetwithprsorted.head()

,Unit ID,Unit Name,Unit name local script,Country/Area,"Subnational unit (province, state)",Status,Status year,Status source,Latitude,Longitude,...,keep,Reserves - Oil,Reserves- Gas,Reserves- Hydrocarbons (unspecified),Production - Oil,Production - Gas,Production - Hydrocarbons (unspecified),"Reserves- Total (Oil, Gas and Hydrocarbons)","Production - Total (Oil, Gas and Hydrocarbons)",Keep?
5668,OG0018000,Patos-Marinza,NaN,Albania,NaN,operating,2021,https://www.offshore-technology.com/marketdata...,40.7705,19.6434,...,yes,29.04,NaN,NaN,4.4836,NaN,NaN,29.040000,4.483600,yes
3496,OG0012600,Adrar,NaN,Algeria,NaN,operating,2023,https://www.woodmac.com/reports/upstream-oil-a...,25.8240,1.7990,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN
3498,OG0012602,Alrar,NaN,Algeria,NaN,operating,2023,https://energycapitalpower.com/largest-gas-fie...,28.5290,9.7900,...,yes,NaN,81070.3667,NaN,NaN,7485.316438,NaN,476.936967,44.036117,yes
3505,OG0012611,Bir Seba Phase 2,NaN,Algeria,NaN,in development,2023,https://www.ogj.com/exploration-development/ar...,32.5240,6.6010,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN
3506,OG0012612,Bir Seba Phase 1,NaN,Algeria,NaN,operating,2021,https://www.ogj.com/exploration-development/ar...,32.5240,6.6010,...,yes,NaN,NaN,NaN,7.3000,NaN,NaN,0.000000,7.300000,yes


In [ ]:
# if Unit type = anything other than asset, field, unit, pool or phase, and there is no production and reserve data as an entry where unit type = project, complex, block, or concession, then sum asset, field, unit, pool or phase production and reserve rows for each project. if there is unit type = project, complex, block, or concession production and reserve data, use those values, ignoring the asset, field, unit, pool or phase production or reserve figures

# Group data by 'Project or complex'
grouped = gogetwithprsorted.groupby('Project or complex')

# Create an empty list to store the results
results = []

# Create counters for project-level and aggregated data
project_level_count = 0
aggregated_count = 0

# Iterate through each project group
for project, group in grouped:
  # Check for project-level production/reserve data
  project_data = group[group['Unit type'].isin(['project', 'complex', 'block', 'concession'])]

  if not project_data.empty:
    # If project-level data exists, use it and skip aggregating lower levels
    project_level_count += 1  # Increment project-level counter
    for index, row in project_data.iterrows():
      results.append({
          'Project or complex': row['Project or complex'],
          'Country/Area': row['Country/Area'],  # Include country for project-level data
          'Gas Reserves': row['Reserves- Gas'],
          'Oil Reserves': row['Reserves - Oil'],
          'Unspecified Hydrocarbons Reserves': row['Reserves- Hydrocarbons (unspecified)'],
          'Gas Production': row['Production - Gas'],
          'Oil Production': row['Production - Oil'],
          'Unspecified Hydrocarbons Production': row['Production - Hydrocarbons (unspecified)'],
          'Total Reserves': row['Reserves- Total (Oil, Gas and Hydrocarbons)'],
          'Total Production': row['Production - Total (Oil, Gas and Hydrocarbons)'],
          'Data Source': 'Project Level' # Tag for project level data
      })

  else:
    # If no project-level data, aggregate lower-level data by country
    aggregated_count += 1  # Increment aggregated counter
    for country, country_group in group[group['Unit type'].isin(['asset', 'field', 'unit', 'pool', 'phase'])].groupby('Country/Area'):
      gas_reserves = country_group['Reserves- Gas'].sum()
      oil_reserves = country_group['Reserves - Oil'].sum()
      unspecified_hydrocarbons_reserves = country_group['Reserves- Hydrocarbons (unspecified)'].sum()
      total_reserves = country_group['Reserves- Total (Oil, Gas and Hydrocarbons)'].sum()
      gas_production = country_group['Production - Gas'].sum()
      oil_production = country_group['Production - Oil'].sum()
      unspecified_hydrocarbons_production = country_group['Production - Hydrocarbons (unspecified)'].sum()
      total_production = country_group['Production - Total (Oil, Gas and Hydrocarbons)'].sum()
      results.append({
          'Project or complex': project,
          'Country/Area': country, # Include country for aggregated data
          'Gas Reserves': gas_reserves,
          'Oil Reserves': oil_reserves,
          'Unspecified Hydrocarbons Reserves': unspecified_hydrocarbons_reserves,
          'Gas Production': gas_production,
          'Oil Production': oil_production,
          'Unspecified Hydrocarbons Production': unspecified_hydrocarbons_production,
          'Total Reserves': total_reserves,
          'Total Production': total_production,
          'Data Source': 'Aggregated'
      })


# Create a new DataFrame from the results
project_summary_df = pd.DataFrame(results)

# Print the counts
print(f"Units with project-level data: {project_level_count}")
print(f"Units with aggregated lower-level data: {aggregated_count}")
print(project_summary_df.shape)
project_summary_df.head()

Units with project-level data: 356
Units with aggregated lower-level data: 711
(1180, 11)


,Project or complex,Country/Area,Gas Reserves,Oil Reserves,Unspecified Hydrocarbons Reserves,Gas Production,Oil Production,Unspecified Hydrocarbons Production,Total Reserves,Total Production,Data Source
0,10097UUU/Athabasca,Canada,NaN,NaN,NaN,NaN,41.391941,NaN,0.0,41.391941,Project Level
1,10103FF/Cold Lake,Canada,NaN,NaN,NaN,NaN,7.553106,NaN,0.0,7.553106,Project Level
2,10159A/Athabasca,Canada,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,Project Level
3,10419Hh/Athabasca,Canada,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,Project Level
4,10419YY/Athabasca,Canada,NaN,NaN,NaN,NaN,17.846682,NaN,0.0,17.846682,Project Level


In [ ]:
# Add unit-only data to project_summary_df
for index, row in gogetwithprsorted[gogetwithprsorted['Project or complex'] == 'Unit Only'].iterrows():
    project_summary_df = pd.concat([project_summary_df, pd.DataFrame({
        'Project or complex': [row['Project or complex']],
        'Unit name': [row['Unit Name']],
        'Country/Area': [row['Country/Area']],
        'Gas Reserves': [row['Reserves- Gas']],
        'Oil Reserves': [row['Reserves - Oil']],
        'Unspecified Hydrocarbons Reserves': [row['Reserves- Hydrocarbons (unspecified)']],
        'Gas Production': [row['Production - Gas']],
        'Oil Production': [row['Production - Oil']],
        'Unspecified Hydrocarbons Production': [row['Production - Hydrocarbons (unspecified)']],
        'Total Reserves': [row['Reserves- Total (Oil, Gas and Hydrocarbons)']],
        'Total Production': [row['Production - Total (Oil, Gas and Hydrocarbons)']],
        'Data Source': ['Unit Level']
    })], ignore_index=True)




In [ ]:
# prompt: project_summary_df to excel

# Assuming project_summary_df is already created as in your provided code.

from google.colab import files

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
excel_file_name = f"project_summary_{timestamp}.xlsx"
project_summary_df.to_excel(excel_file_name, index=False)

files.download(excel_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# # prompt: add all none project associated units from to filtered_main_data to project_summary_df and fill out the columns in that table based on the data in filtered_main_data

# # Filter for units tagged as "Unit Only"
# non_project_units = gogetwithprsorted['Project or complex'] == 'Unit Only'

# # Create a list to store the new rows
# new_rows = []

# for index, row in non_project_units.iterrows():
#     new_row = {
#         'Project or complex': row['Project or complex'],
#         'Unit name': row['Unit Name'],
#         'Country/Area': row['Country/Area'],
#         'Gas Reserves': row['Reserves- Gas'],
#         'Oil Reserves': row['Reserves - Oil'],
#         'Unspecified Hydrocarbons Reserves': row['Reserves- Hydrocarbons (unspecified)'],
#         'Gas Production': row['Production - Gas'],
#         'Oil Production': row['Production - Oil'],
#         'Unspecified Hydrocarbons Production': row['Production - Hydrocarbons (unspecified)'],
#         'Total Reserves': row['Reserves- Total (Oil, Gas and Hydrocarbons)'],
#         'Total Production': row['Production - Total (Oil, Gas and Hydrocarbons)'],
#         'Data Source': 'Unit Level' # Tagging the source as unit level
#     }
#     new_rows.append(new_row)

# # Create a DataFrame from the new rows
# new_units_df = pd.DataFrame(new_rows)

# # Concatenate the new DataFrame with project_summary_df
# project_summary_df = pd.concat([project_summary_df, new_units_df], ignore_index=True)

# project_summary_df.shape

AttributeError: 'Series' object has no attribute 'iterrows'

In [ ]:
# prompt: Make two DF, one with the counts and a project_summary_df.

# Create the counts DataFrame
counts_df = pd.DataFrame({
    'Data Source': ['Units with project-level data', 'Units with aggregated lower-level data', "Units with stand alone data"],
    'Count': [project_level_count, aggregated_count, blank_project_count]
})

print("Counts DataFrame:")
print(counts_df)
print("\nProject Summary DataFrame:")
print(project_summary_df.head())

Counts DataFrame:
                              Data Source  Count
0           Units with project-level data    356
1  Units with aggregated lower-level data    711
2             Units with stand alone data   5064

Project Summary DataFrame:
   Project or complex Country/Area  Gas Reserves  Oil Reserves  \
0  10097UUU/Athabasca       Canada           NaN           NaN   
1   10103FF/Cold Lake       Canada           NaN           NaN   
2    10159A/Athabasca       Canada           NaN           NaN   
3   10419Hh/Athabasca       Canada           NaN           NaN   
4   10419YY/Athabasca       Canada           NaN           NaN   

   Unspecified Hydrocarbons Reserves  Gas Production  Oil Production  \
0                                NaN             NaN       41.391941   
1                                NaN             NaN        7.553106   
2                                NaN             NaN             NaN   
3                                NaN             NaN             NaN   
4

In [ ]:
!pip install xlsxwriter

import xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
#save out complied sheet with Reserves, counts_df, and Project Summary DataFrame as one excel sheet

# time stamp generation
from datetime import datetime

# Get the current date and time
now = datetime.now()

# Format the date and time as "GOGET as of {timestamp_str} with PR"
timestamp_str = now.strftime("%Y_%m_%d_%H:%M")

print(timestamp_str)

2025_02_18_18:23


In [ ]:
counts_df

,Data Source,Count
0,Units with project-level data,356
1,Units with aggregated lower-level data,711
2,Units with stand alone data,5064


In [ ]:
import os

In [ ]:
# prompt: save counts_df project_summary_df and filtered_main_data_df to excel sheets
# Create the necessary directories if they don't exist
output_dir = os.path.dirname(output_file)  # Get the directory part of the path
os.makedirs(output_dir, exist_ok=True)     # Create the directory (and any parent directories) if they don't exist


# Create a Pandas Excel writer using XlsxWriter as the engine.
output_file = f'/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Compiled Versions of GOGET/GOGET as of {timestamp_str} with PR grouped by project/complex.xlsx'
writer = pd.ExcelWriter(output_file, engine='xlsxwriter')

# Write each DataFrame to a different worksheet.
filtered_main_data_df.to_excel(writer, sheet_name='Main Data', index=False)
counts_df.to_excel(writer, sheet_name='Counts', index=False)
project_summary_df.to_excel(writer, sheet_name='Project Summary', index=False)

# Close the Pandas Excel writer and output the Excel file.
writer.close()

NameError: name 'output_file' is not defined

In [ ]:
print("output saved at " + output_file)

NameError: name 'output_file' is not defined